# TP17 - BlueVisionary

# Iteration 2 - Data Cleaning and Wrangling

In [ ]:
# Install Libraries
#!pip install pandas
#!pip install numpy
#!pip install csv
#!pip install geojson
#!pip install matplotlib
#!pip install os
#!pip install math

In [ ]:
# importing libraries

import csv
import os
import json
import matplotlib as plt
import geojson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import pi

In [ ]:
# Get the current working directory (directory of the script)
current_dir = os.getcwd()

# Construct the relative path to the CSV file
polymer_df = os.path.join(current_dir, "AODN-IMOS Microdebris Data (ALL) from Jan-2021 to Apr-2024.csv")

# Reading processing facility data from CSV
polymer_df_1 = pd.read_csv(polymer_df , encoding='unicode_escape')

In [ ]:
polymer_df_1.head()

In [ ]:
polymer_df_1.info()

In [ ]:
# List of unecessary columns
drop_cols = ["ORGANISATION", "REGION", "SAMPLE_DATE", "START_TIME", "END_TIME", "LOCAL_TIME_ZONE", "BIOFOULING", "FIELD_SAMPLE_ID", "ITEM_ID", 
"PHOTO_ID", "LAB_SAMPLE_ID" ,"REGION", "VESSEL","SAMPLE_DAY"]

In [ ]:
# Dropping unecessary columns
polymer_df_1.drop(columns = drop_cols, inplace= True)

In [ ]:
# Checking null values
polymer_df_1.isnull().sum()

In [ ]:
# Dropping columns with more than 50% null values
drop_cols_null = ["WIDTH_MM"]
polymer_df_1.drop(columns = drop_cols_null, inplace= True)

In [ ]:
# Replacing NaN values in all numerical columns with column average/mean
for num_column in ["LENGTH_MM", "AREA_MM2"]:
    polymer_df_1[num_column].fillna(polymer_df_1[num_column].mean(), inplace=True)

In [ ]:
# Replacing NaN values in the cateorical columns with column mode
for cat_column in ['POLYMER_TYPE', 'POLYMER_CATEGORY', 'SHAPE', 'PRIMARY_VS_SECONDARY','COLOUR','POTENTIAL_SOURCE', 'ORGANISM']:
    polymer_df_1[cat_column].fillna(polymer_df_1[cat_column].mode()[0], inplace=True)

In [ ]:
# Again checking the null values
polymer_df_1.isnull().sum()

In [ ]:
polymer_df_1.head()

In [ ]:
# Printing unique values in a "POTENTIAL_SOURCE"column
print(polymer_df_1.POTENTIAL_SOURCE.unique())

In [ ]:
# Replacing a list of values with one value
polymer_df_1['POTENTIAL_SOURCE'].replace(['household plastics',
                           'Thermoplastic. General household. Plastic bags, containers, batteries, automotive or instrument parts.',
                           'Thermoplastic. General household. Plastic bags, containers, batteries, automotive and instrument parts.',
                           'Thermoplastic. Industrial applications, textiles and general household.',
                           'Thermoplastic. Industrial applications and general household.'],
                           'Thermoplastic. General household. Plastic bags to heavy duty plastic containers.', inplace=True)

polymer_df_1['POTENTIAL_SOURCE'].replace(['Thermoplastic. Industrial applications, textiles and general household.',
                           'textile industry',
                           'Thermoplastic. Industrial applications, textiles and general household.',
                           'Thermoplastic. Industrial applications or textiles.',
                           'Protein. Textiles or clothing.',
                           'Cellulosic. Textiles and clothing',
                           'Thermoplastic. Textiles and clothing',
                           'Thermoplastic. Textiles and clothing.',
                           'Thermoplastic. Industrial applications and textiles.'],
                           'Thermoplastic. Industrial applications, textiles and clothing.', inplace=True)

                    
polymer_df_1['POTENTIAL_SOURCE'].replace(['Thermoplastic. Cosmetical or pharmaceutical industry. Exfoliants.',
                                     'cosmetical and pharmaceutical industry',
                                     'Surfactants, waxes for thermoplastics. Cosmetical or Pharmaceutical industry.',
                                     'Surfactants, waxes for thermoplastics. Industrial application, food safe products, cosmetical or cpharmaceutical industry.'],
                                     'Thermoplastic. Surfactants, waxes, cosmetical or pharmaceutical industry', inplace=True)

polymer_df_1['POTENTIAL_SOURCE'].replace(['paints and coatings',
                                     'Thermoplastic. Paint and resin.',
                                     'Thermoplastic. Resins, paints and adhesives',
                                     'Thermoplastic. Paint, resin and textile.',
                                     'Thermoplastic or thermoset. Paint and resin.'],
                                     'Thermoplastic. Resins, coatings, paints and adhesives', inplace=True)

polymer_df_1['POTENTIAL_SOURCE'].replace(['Additive or plasticizer for fire-resistant/retardant polymers for building and construction',
                                          'Additives in paints, basis for hard glass, tiles and surface.',
                                          'Thermoplastic. Building and construction, industrial applicatios and general household. ',
                                          'Thermoset. Building and construction industry. Seals or water-proof liners.'],
                                          'Thermoplastic. Building and construction. Paints, hard glass, tiles and surface', inplace=True)
                                          

In [ ]:
poly_cleaned_df = polymer_df_1[['IMOS_CODE','STATE_TERRITORY','STATION_NAME',
                                            'TOW_REPLICATE','START_LAT',
                                            'START_LONG','END_LAT','END_LONG','SAMPLE_YEAR','SAMPLE_MONTH',
                                            'POLYMER_CATEGORY','POLYMER_TYPE','POTENTIAL_SOURCE','SHAPE',
                                            'COLOUR']].copy()

In [ ]:
poly_cleaned_df.head()

In [ ]:
# Write file to csv
poly_cleaned_df.to_csv('polymer_main.csv', index=False)

In [ ]:
# aggregagation of data based on state and year 
aggregated_data_1 = poly_cleaned_df.groupby(['STATE_TERRITORY', 'SAMPLE_YEAR']).size().reset_index(name='POLYMER_COUNT')


In [ ]:
aggregated_data_1.head(30)

In [ ]:
# write to csv file
aggregated_data_1.to_csv('State_poly_count_year.csv', index=False)

In [ ]:
#filter dataframe based on top six polymer types
poly_cleaned_df['POLYMER_TYPE'].value_counts().head(6)

In [ ]:
 # list of top five polymer type 
top_6_polymers = ["polyethylene", "polypropylene", "polyethylene glycol", "thermoplastic", "thermoset", "polystyrene"]

In [ ]:
# creating a new column
poly_cleaned_df['POLYMER_TYPE_NEW'] = poly_cleaned_df['POLYMER_TYPE'].apply(lambda x: x if x in top_6_polymers else 'other')

In [ ]:
poly_cleaned_df.shape

In [ ]:
poly_cleaned_df.head()

In [ ]:
# group by state, polymer type, polymer source, and year
aggregated_data_2 = poly_cleaned_df.groupby(['STATE_TERRITORY','SAMPLE_YEAR', 'POLYMER_TYPE_NEW']).size().reset_index(name='POLYMER_COUNT')

In [ ]:
aggregated_data_2.head(50)

In [ ]:
# write to csv 
aggregated_data_2.to_csv('State_polytype_count_year.csv', index=False)

In [ ]:
# plotting the trend chart based on aggregation 1 df
aggregated_data_1_trend = aggregated_data_1

# Get the list of unique states
states = aggregated_data_1['STATE_TERRITORY'].unique()



In [ ]:
# Plot current trends for each state
for state in states:
    # Filter data for the current state
    state_data = aggregated_data_1[aggregated_data_1['STATE_TERRITORY'] == state]
    
    # Plotting the trend
    plt.figure(figsize=(8, 6))
    plt.plot(state_data['SAMPLE_YEAR'], state_data['POLYMER_COUNT'], marker='o', linestyle='-', label=state)
    plt.title(f'Trend of Polymer Count for {state}')
    plt.xlabel('Year')
    plt.ylabel('Polymer Count')
    plt.grid(True)
    plt.legend()
    
    # Display the plot
    plt.show()

In [ ]:
# Calculate year-over-year percentage change for each state, then project to 2030

# Empty list to store projected data
extended_data = []

# Iterate over each state
for state in states:
    # Filter data for the current state
    state_data = aggregated_data_1[aggregated_data_1['STATE_TERRITORY'] == state].copy()
    
    # Sort data by year
    state_data = state_data.sort_values('SAMPLE_YEAR')
    
    # Calculate YoY change for the current state
    state_data['YoY Change (%)'] = state_data['POLYMER_COUNT'].pct_change() * 100
    
    # Calculate average YoY change
    average_yoy_change = state_data['YoY Change (%)'].mean()
    
    # Get the last available year and polymer count
    last_year = state_data['SAMPLE_YEAR'].max()
    last_value = state_data[state_data['SAMPLE_YEAR'] == last_year]['POLYMER_COUNT'].values[0]
    
    # Project values from the next year up to 2030
    for year in range(last_year + 1, 2031):
        # Project the polymer count using the average YoY change
        projected_value = last_value * (1 + average_yoy_change / 100)
        extended_data.append([state, year, projected_value])
        # Update last_value for next iteration
        last_value = projected_value

# Create a DataFrame for the extended data
extended_df = pd.DataFrame(extended_data, columns=['STATE_TERRITORY', 'SAMPLE_YEAR', 'POLYMER_COUNT'])

# Combine the original data with the extended data
full_df = pd.concat([aggregated_data_1[['STATE_TERRITORY', 'SAMPLE_YEAR', 'POLYMER_COUNT']], extended_df], ignore_index=True)


In [ ]:
# Plot trend charts for each state including projections until 2030
for state in states:
    # Filter data for the current state
    state_data = full_df[full_df['STATE_TERRITORY'] == state]
    
    # Plotting the trend
    plt.figure(figsize=(8, 6))
    plt.plot(state_data['SAMPLE_YEAR'], state_data['POLYMER_COUNT'], marker='o', linestyle='-', label=state)
    plt.title(f'Trend of Polymer Count for {state} (Extended to 2030)')
    plt.xlabel('Year')
    plt.ylabel('Polymer Count')
    plt.grid(True)
    plt.legend()
    
    # Display the plot
    plt.show()


In [ ]:
# write data to csv file
full_df.to_csv('State_poly_count_trends2030.csv', index=False)

In [ ]:
# aagregation based on potential common sources 
aggregated_data_3_new = poly_cleaned_df.groupby(['POLYMER_TYPE_NEW', 'POTENTIAL_SOURCE']).size().reset_index(name='POLYMER_COUNT')

aggregated_data_3_new.head(40)

In [ ]:
# Filter for specific state and year (for example NSW and 2022)
filtered_df = aggregated_data_2[(aggregated_data_2['STATE_TERRITORY'] == 'NSW') & (aggregated_data_2['SAMPLE_YEAR'] == 2023)]

# Prepare data for radar chart
categories = filtered_df['POLYMER_TYPE'].tolist()
values = filtered_df['POLYMER_COUNT'].tolist()

# Complete the polygon by repeating the first value
values += values[:1]

# Radar chart setup
num_vars = len(categories)

# Calculate angle of each axis
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]

# Create radar chart figure
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw the outline of the radar chart
ax.fill(angles, values, color='blue', alpha=0.25)
ax.plot(angles, values, color='blue', linewidth=2)

# Add labels to the chart
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)

# Display the radar chart
plt.title('Polymer Types in NSW (2022)')
plt.show()
plt.show()

# Pollution Severity

In [ ]:
#Dataframe microplastic: 

#pollution_serverity 

state= ["NSW", "NT", "QLD", "SA", "TAS", "VIC", "WA"]
count=[759, 35, 545, 418, 36, 65, 270]
marine_area_kmsqaure= [8802, 71839, 121994, 60032, 22357, 10213, 115740]
law_count= [2,3,1,1,4,1,3]

dict = {'state': state, 'polymer_count': count, 'marine_area_kmsqaure': marine_area_kmsqaure, 'law_count': law_count } 
pollution_serverity = pd.DataFrame(dict)

In [ ]:
pollution_serverity.head(10)

In [ ]:
pollution_serverity["severity_count_per_km_sq"] = pollution_serverity["polymer_count"]/ pollution_serverity["marine_area_kmsqaure"]
pollution_serverity["score"] = [0,0,0,0,0,0,0]

In [ ]:
pollution_serverity.head(10)

In [ ]:
# writing file to json
pollution_serverity.to_json('pollution_severity.json', orient='records', indent=4)

In [ ]:
# Also writing to csv
pollution_serverity.to_csv('pollution_severity.csv')